<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.16 Build Your Own Model

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 1. Introduction

This is the project notebook of the Mobile AI module. Everything you have studied piece by piece comes together here, in one start-to-finish pipeline that you configure, run, and defend: **data → tokenizer → parameters → training → evaluation**. At the end you own a working language model: a very small one, but genuinely yours, with weights no one else has.

Every stage of the pipeline is one you have already met:

| Stage | Where you learned it |
|:------|:---------------------|
| Words become token IDs | [`05.01_Understanding_Tokenization.ipynb`](./05.01_Understanding_Tokenization.ipynb) |
| Token IDs become vectors | [`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb) |
| Vectors attend to each other | [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb) |
| Vectors become next-token probabilities | [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb) |
| Gradient descent fills in the weights | [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) |

The model you build is a **MicroLLM**: a word-level GPT with a few thousand parameters, small enough that a full training run takes a few minutes of laptop CPU and every single matrix can be looked at. The heavy lifting (the autograd engine, the forward pass, the plots) lives in a helper module so that this notebook stays about *decisions*, not code. Your work is choosing the configuration, reading the evidence, and explaining what you see.

#### 1.1 What this project asks of you

<div class="alert alert-block alert-info">

**This notebook is a graded homework project.** The deliverable is this file, run from top to bottom with **your** configuration, all outputs visible and all questions answered with evidence from your own run.:

| What is graded | Weight |
|:---------------|:------:|
| A complete top-to-bottom run with your configuration and visible outputs | 40 % |
| Question answers anchored in your own numbers and samples | 40 % |
| Your experiment (Section 8): one deliberate change, measured and explained | 20 % |

There is no single correct configuration. What is graded is the quality of your evidence and reasoning, not the outcome.
</div>

### Environment Setup

Run this cell first. It makes the MicroLLM helper code in `../../.admin/notebooks/microllm/` importable and installs any missing visualization packages.

In [ ]:
from pathlib import Path
import sys

# Locate the shared helpers from the notebook folder or repository root.
_helper_dir = next(
    (root / ".admin/notebooks/microllm"
     for root in (Path.cwd(), *Path.cwd().parents)
     if (root / ".admin/notebooks/microllm/micro_llm_helpers.py").is_file()),
    None,
)
if _helper_dir is None:
    raise FileNotFoundError(
        "MicroLLM helpers were not found. Open this notebook inside the full "
        "course repository, including .admin/notebooks/microllm."
    )
if str(_helper_dir) not in sys.path:
    sys.path.insert(0, str(_helper_dir))

from micro_llm_helpers import ensure_notebook_environment

ensure_notebook_environment()

### 2. Your Configuration

Every decision of this project is collected in one place, so you always know exactly what you are running. The defaults produce a working model in a few minutes. Read through the comments, then leave the values as they are for your **baseline run**. You will come back to change *one* of them in Section 8.

| Knob | What it controls | Studied in |
|:-----|:-----------------|:-----------|
| `n_layer` | how many transformer layers process each token | [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb) |
| `n_embd` | the width $d$ of every token vector | [`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb) |
| `n_head` | how many attention heads share that width | [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb) |
| `block_size` | the context length $T$: how far back the model can look | [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb) |
| `num_steps` | how long we train | [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) |
| `learning_rate` | how large each training nudge is | [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) |
| `seed` | makes your run reproducible | — |

In [ ]:
CONFIG = {
    # ---- model architecture -------------------------------------------------
    "n_layer":    1,      # transformer layers                (try 1, 2, or 3)
    "n_embd":     16,     # embedding width d                 (try 8, 16, or 32)
    "n_head":     4,      # attention heads                   (must divide n_embd)
    "block_size": 16,     # context length T in tokens

    # ---- training -----------------------------------------------------------
    "num_steps":     300,     # 300 steps take a few minutes of laptop CPU
    "learning_rate": 0.01,    # the size of each nudge

    # ---- reproducibility ----------------------------------------------------
    "seed": 2,                # fixes data shuffling and weight initialization
}

# Optional (Section 8): extend the training data with your own sentence family.
# Use simple lowercase words only: every word you introduce becomes a new token.
MY_SENTENCES = [
    # "boil water for the sweet tea",
    # "pour the tea before it is cold",
]

print("configuration loaded:", CONFIG)
print(f"custom sentences: {len(MY_SENTENCES)}")

### 3. Data and Tokenizer

The training data is a tiny corpus of thirty baking-recipe sentences (plus whatever you add in `MY_SENTENCES`). As in [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb), the smallness is the point: a model can only learn what its data contains, and with thirty sentences you can *know* the data completely, which lets you judge later whether a generated sentence was memorized, recombined, or invented.

The tokenizer here works at the **word level**: every distinct lowercase word becomes one token ID. One special token, `BOS` (Beginning Of Sequence), is wrapped around every sentence and doubles as the stop signal during generation:

| | | | | | |
|:--|:--:|:--:|:--:|:--:|:--:|
| **raw sentence** | | mix | flour | and | sugar |
| **training tokens** | `BOS` | `mix` | `flour` | `and` | `sugar` → `BOS` |
| **token IDs** | 70 | 36 | 21 | 3 | 54 → 70 |

Word tokenization is chosen because it is easy to inspect: every ID maps to one visible word. Production LLMs use **subword** tokenizers instead, for the reasons you explored in [`05.01_Understanding_Tokenization.ipynb`](./05.01_Understanding_Tokenization.ipynb): word-level vocabularies explode with rare words and cannot represent anything unseen, while subwords compose unfamiliar words from familiar pieces.

In [ ]:
import random
import re

from micro_llm_training_data import TRAINING_SENTENCES

training_examples = TRAINING_SENTENCES + MY_SENTENCES
random.Random(CONFIG["seed"]).shuffle(training_examples)

WORD_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")

def split_into_words(text):
    return WORD_RE.findall(text.lower())

id_to_token = sorted({word for example in training_examples for word in split_into_words(example)})
token_to_id = {token: i for i, token in enumerate(id_to_token)}
BOS = len(id_to_token)                 # the special beginning-of-sequence token
vocab_size = len(id_to_token) + 1

def encode(text):
    return [token_to_id[word] for word in split_into_words(text)]

def decode(token_ids):
    return " ".join(id_to_token[token_id] for token_id in token_ids)

print(f"training sentences : {len(training_examples)}")
print(f"vocabulary size    : {vocab_size} ({len(id_to_token)} words + BOS)")

example = training_examples[0]
print(f"\nworked example")
print(f"  raw sentence : {example!r}")
print(f"  word tokens  : {split_into_words(example)}")
print(f"  token IDs    : {encode(example)}")
print(f"  decoded back : {decode(encode(example))!r}")

### 4. The Model: Seven Kinds of Matrices

A GPT-style model is nothing more than a handful of weight matrices and a fixed recipe for multiplying them. The fixed settings from your configuration determine every shape: $V$ is the vocabulary size, $d$ = `n_embd` is the embedding width, and $T$ = `block_size` is the context length. The matrices are the complete inventory of what training can change:

| Matrix | Shape | Role | Where you met it |
|:-------|:------|:-----|:-----------------|
| `wte` | $V \times d$ | token embedding: which word is this? | [`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb) |
| `wpe` | $T \times d$ | position embedding: where does it appear? | [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb) |
| `attn_wq`, `attn_wk`, `attn_wv` | $d \times d$ | queries, keys, and values for attention | [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb) |
| `attn_wo` | $d \times d$ | mixes the attention result back together | [`05.03_Understanding_Attention.ipynb`](./05.03_Understanding_Attention.ipynb) |
| `mlp_fc1`, `mlp_fc2` | $4d \times d$, $d \times 4d$ | per-token feature processing, with a ReLU in between | [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) |
| `lm_head` | $V \times d$ | turns the final vector into a score per vocabulary word | [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb) |

Each additional layer repeats the attention and MLP matrices once more. Every entry starts as a small random number and is stored as a `Value` object: a scalar that remembers how it was computed, so that gradients can flow back through it during training.

> The weights here are kept as full floating-point values so the training math stays inspectable. A deployed model would quantize them to 8 or 4 bits, the trade you measured in [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb).

In [ ]:
from micro_llm_helpers import Value, show_parameter_table

random.seed(CONFIG["seed"])

n_layer, n_embd = CONFIG["n_layer"], CONFIG["n_embd"]
n_head, block_size = CONFIG["n_head"], CONFIG["block_size"]
head_dim = n_embd // n_head

def matrix(nout, nin, std=0.08):
    return [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

state_dict = {
    "wte": matrix(vocab_size, n_embd),
    "wpe": matrix(block_size, n_embd),
    "lm_head": matrix(vocab_size, n_embd),
}
for i in range(n_layer):
    state_dict[f"layer{i}.attn_wq"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wk"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wv"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.attn_wo"] = matrix(n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc1"] = matrix(4 * n_embd, n_embd)
    state_dict[f"layer{i}.mlp_fc2"] = matrix(n_embd, 4 * n_embd)

params = [p for mat in state_dict.values() for row in mat for p in row]

show_parameter_table(state_dict)

These are your trainable parameters: a few thousand of them, against the few *billion* of the models on your Raspberry Pi. The heatmaps below draw every matrix as a small image: each colored square is one weight, on a fixed color scale from $-1$ (blue) through $0$ (white) to $+1$ (red). At initialization everything is faint noise near zero. Remember this picture: you will see these matrices again after training.

In [ ]:
from micro_llm_helpers import show_parameter_heatmaps

show_parameter_heatmaps(state_dict)

### 5. Training

You watched this loop live in [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb), and the same four steps repeat here: **draw** one training sentence, **predict** every next token, **measure** the surprise, **nudge** every weight downhill. Two refinements are worth knowing about, because your configuration controls them:

- **Adam** is the optimizer that performs the nudging. For every parameter it keeps two running memories: `m`, the recent direction of its gradients, and `v`, how large and noisy they have been. It combines them into a steadier per-parameter step size than raw gradient descent would take. It is the de-facto standard optimizer of deep learning [04].
- **Learning-rate decay**: the step size starts at your `learning_rate` and shrinks linearly to zero over the run: large corrective steps early, fine adjustments at the end.

One unit note: the loss here uses the natural logarithm (*nats* per **word**, since our tokens are words). The bits-per-character story from [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) is the same idea up to a constant factor. Your anchor for a clueless model is $\ln V$. For the default vocabulary of 71, that is about **4.26**. Watch where the first logged loss starts.

The loop itself is called as one function here. Expect a few minutes: the model is written in pure, inspectable Python rather than an optimized library, so *you* can read every line of what it does, at the price of it being thousands of times slower per weight than the tensor libraries real training uses.

In [ ]:
import time

from micro_llm_helpers import make_gpt_forward, softmax, train_microllm

gpt_forward = make_gpt_forward(state_dict, n_layer, n_head, head_dim)

start_time = time.time()
training_log = train_microllm(
    params,
    training_examples,
    BOS,
    encode,
    block_size,
    n_layer,
    gpt_forward,
    learning_rate=CONFIG["learning_rate"],
    num_steps=CONFIG["num_steps"],
)
minutes = (time.time() - start_time) / 60
print(f"trained {CONFIG['num_steps']} steps in {minutes:.1f} minutes on this machine")

from micro_llm_helpers import show_training_log

show_training_log(training_log, CONFIG["num_steps"], len(params))

#### 5.1 The same matrices, after training

The heatmaps below use the same fixed color scale as before. Compare them with the initialization picture from Section 4: the optimizer has pushed many weights visibly away from zero. Nothing else happened during training: *this color change is the entire result of the last few minutes*.

In [ ]:
show_parameter_heatmaps(state_dict)

#### 5.2 What the embeddings learned

In [`05.02_Understanding_Word_Embedding.ipynb`](./05.02_Understanding_Word_Embedding.ipynb) you projected the embedding space of a professionally trained model with PCA. Here is the same visualization for **your own** token embedding matrix `wte`, fresh from training. Rotate it and look for structure: words that play the same role in the recipes (ingredients, actions, the cake/cookie pair) should drift toward each other, because tokens that are interchangeable in context receive similar gradient pushes. With a corpus this small the clusters are rough, which is itself worth noticing.

In [ ]:
from micro_llm_helpers import show_token_embedding_pca

show_token_embedding_pca(state_dict, id_to_token, BOS)

### 6. Evaluation: What Did We Actually Build?

Time to grade the model: first with fixed printed samples (so your submitted notebook contains the evidence), then interactively.

#### 6.1 Printed completions

The cell below is the sampling loop from [`05.05_Introduction_to_LLMs.ipynb`](./05.05_Introduction_to_LLMs.ipynb) in miniature: feed the prompt through the model, turn the logits for the next word into probabilities with `softmax`, sample one word, repeat, stopping at `BOS` or after twelve new words. The `temperature` divides the logits before softmax, exactly as in [`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb): low values pick the safest word, high values gamble.

While you read the outputs, keep the thirty training sentences from Section 3 in mind and ask of each completion: **memorized, recombined, or invented?**

In [ ]:
import math

def complete(prompt, temperature=0.6, max_new_words=12, sample_seed=0):
    """Complete a prompt with the trained model, one sampled word at a time."""
    rng = random.Random(sample_seed)
    token_ids = [BOS] + encode(prompt)
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    for position, token_id in enumerate(token_ids[:-1]):
        gpt_forward(token_id, position, keys, values)     # warm up on the prompt

    generated = []
    token_id = token_ids[-1]
    for position in range(len(token_ids) - 1, min(block_size, len(token_ids) - 1 + max_new_words)):
        logits = gpt_forward(token_id, position, keys, values)
        weights = [math.exp(logit.data / temperature) for logit in logits]
        token_id = rng.choices(range(vocab_size), weights)[0]
        if token_id == BOS:
            break
        generated.append(id_to_token[token_id])
    return f"{prompt} -> {' '.join(generated)}"

PROMPTS = ["mix flour and", "the oven makes", "bake the"]

for prompt in PROMPTS:
    for sample_seed in (0, 1):
        print(complete(prompt, temperature=0.6, sample_seed=sample_seed))
print()
print("same prompt, cold versus hot sampling:")
print("  t=0.2 :", complete(PROMPTS[0], temperature=0.2))
print("  t=1.3 :", complete(PROMPTS[0], temperature=1.3))

#### 6.2 Interactive completions

The widget below lets you probe the model freely. It shows several sampled continuations at once. The percentage next to each option is the product of the model's probabilities for the generated words: an internal confidence score for that continuation, not a real-world probability of being correct. Prompts must use words from the training vocabulary and the widget warns you about unknown words.

In [ ]:
from micro_llm_helpers import show_completion_widget

show_completion_widget(
    split_into_words,
    token_to_id,
    id_to_token,
    BOS,
    block_size,
    n_layer,
    gpt_forward,
    softmax,
    vocab_size,
)

### 7. Ship It

A trained model is, in the end, a file. The cell below saves yours (configuration, tokenizer, and every trained weight) and reports what it weighs on disk. Since the file is plain JSON and the model is plain Python, it would run anywhere Python runs, your Raspberry Pi included.

Notice the storage arithmetic: JSON text spends tens of bytes per weight (the cell prints the exact figure for your run). The formats you studied in [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb) spend 2 bytes (FP16) or half a byte (INT4): your first compression decision, waiting to be made.

In [ ]:
from micro_llm_helpers import MICROLLM_TRAINED_STATE_PATH, save_microllm_state

trained_state = {
    "training_examples": training_examples,
    "id_to_token": id_to_token,
    "BOS": BOS,
    "n_layer": n_layer,
    "n_embd": n_embd,
    "block_size": block_size,
    "n_head": n_head,
    "head_dim": head_dim,
    "state_dict": state_dict,
}
saved_path = save_microllm_state(MICROLLM_TRAINED_STATE_PATH, trained_state, training_log)

size_kb = saved_path.stat().st_size / 1e3
print(f"saved   : {saved_path}")
print(f"weights : {len(params):,} trainable values")
print(f"size    : {size_kb:.0f} KB  ({saved_path.stat().st_size / len(params):.0f} bytes per weight as JSON text)")

### 8. Your Experiment

This is the graded core of the project. You have a baseline run. Now change **exactly one thing**, run the notebook again from top to bottom, and defend what you find with numbers and samples. Pick one:

| Change | Edit in Section 2 | What to watch |
|:-------|:------------------|:--------------|
| **Deeper** | `n_layer: 1 → 2` | parameter count, final loss, training time |
| **Wider** | `n_embd: 16 → 32` | parameter count, final loss, completion quality |
| **Longer training** | `num_steps: 300 → 600` | the shape of the loss curve, memorization |
| **Braver or more careful nudges** | `learning_rate: 0.01 → 0.03` or `→ 0.003` | stability and speed of the loss curve |
| **Your own data** | add 3–6 sentences to `MY_SENTENCES` | vocabulary size, completions using your new words |

The procedure (treat it like the lab it is):

1. **Record your baseline** before changing anything: final loss, training time, parameter count, and two or three representative completions (they are printed in Section 6.1; copy them into Question 11 now).
2. Make your one change in Section 2, then run everything again from the top (*Kernel → Restart Kernel and Run All Cells*).
3. **Record the same numbers again**, and answer Question 11 with the before/after comparison and your explanation.

<div class="alert alert-block alert-info">

One change, honestly measured, beats five changes eyeballed. If your change makes the model *worse*, that is a perfectly good result: explain why, and you have done exactly what this section asks.
</div>

### 9. Summarizing Remarks

Step back and look at what this notebook actually contains, because it is easy to miss while running cells: a complete language model, built end to end from parts you understand individually. Thirty sentences became token IDs, while seven kinds of matrices turned token IDs into next-word probabilities. A few hundred repetitions of predict-measure-nudge turned random noise into a model that speaks passable recipe English, and you can point at every weight that changed and at the exact sentences responsible.

Everything separating your MicroLLM from the models on your Raspberry Pi is quantity, not kind: more sentences, wider matrices, more layers, more steps, plus the post-training stages from [`05.06_Understanding_Training.ipynb`](./05.06_Understanding_Training.ipynb) that turn a continuation machine into an assistant, and the compression from [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb) that fits the result into a pocket. When a vendor next shows you a model, you now know what is inside the box: matrices, shaped by data, judged by surprise.

### APPENDIX

#### Acknowledgement

The MicroLLM implementation is adapted from Andrej Karpathy's compact `microgpt.py` example. Original author: Andrej Karpathy. Reference: [https://gist.github.com/karpathy](https://gist.github.com/karpathy). You can also find Andrej Karpathy's videos on his [YouTube channel](https://www.youtube.com/@AndrejKarpathy).

#### Related Projects

[01] [`micrograd`](https://github.com/karpathy/micrograd): a tiny scalar-valued autograd engine and neural-network teaching project, and the ancestor of the `Value` class used here.

[02] [`makemore`](https://github.com/karpathy/makemore): an autoregressive character-level language model for generating more examples like a text dataset.

[03] [`nanoGPT`](https://github.com/karpathy/nanoGPT): a compact GPT training and fine-tuning codebase, and the natural next step up from this notebook.

[04] [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980): the original Adam optimizer paper by Diederik P. Kingma and Jimmy Ba. Adam is an influential and widely used optimizer; many modern systems also use close variants such as AdamW.

---
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. Many questions ask for concrete values or outputs from your own notebook run; generic answers without run-specific evidence may receive reduced credit. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Question 01: Configuration**

Report your baseline configuration: `n_layer`, `n_embd`, `n_head`, `block_size`, `num_steps`, `learning_rate`, and the vocabulary size $V$ of your run.

Answer:

 - (add your short answer here)

**Question 02: Tokenizer**

Copy the worked example from Section 3: one raw sentence, its word tokens, and its token IDs. What is the purpose of the `BOS` token, and where does it appear during training?

Answer:

 - (add your short answer here)

**Question 03: Parameters**

From the parameter table in Section 4: which matrix holds the most trainable values in your configuration, and why does its shape depend on the vocabulary size? Report the total parameter count.

Answer:

 - (add your short answer here)

**Question 04: Before and after**

Compare the heatmaps from Section 4 with those in Section 5.1. Describe one matrix whose appearance changed clearly. What single process caused every one of those color changes?

Answer:

 - (add your short answer here)

**Question 05: The loss**

Report your first and final logged loss. The first one should sit near $\ln V$. Explain why that is the expected starting point for an untrained model.

Answer:

 - (add your short answer here)

**Question 06: A bump in the curve**

Find a point in your training log where the loss briefly *increases*. Why can this happen even when training is working correctly?

Answer:

 - (add your short answer here)

**Question 07: Embedding space**

Rotate the PCA plot in Section 5.2 and name one pair or group of words that ended up close together. Why would training push exactly these words toward each other?

Answer:

 - (add your short answer here)

**Question 08: Temperature**

Compare the cold (`t=0.2`) and hot (`t=1.3`) completions from Section 6.1, or your own from the widget. Describe the difference, and connect it to the temperature experiments you ran on the Raspberry Pi in [`05.11_Understanding_Inference_Control.ipynb`](./05.11_Understanding_Inference_Control.ipynb).

Answer:

 - (add your short answer here)

**Question 09: Memorized, recombined, or invented?**

Pick one completion that looks plausible and one that looks wrong or strange. For each, point to the training sentence(s) that most likely produced it, and classify it: memorized, recombined, or invented.

Answer:

 - (add your short answer here)

**Question 10: The file**

Report the size of your saved model file and its bytes per weight. Using the formats from [`05.08_Making_Models_Smaller.ipynb`](./05.08_Making_Models_Smaller.ipynb), roughly what would the same model weigh in FP16 and in INT4?

Answer:

 - (add your short answer here)

**Question 11: Your experiment (graded core)**

State the one change you made. Report before and after: final loss, training time, parameter count (if it changed), and one representative completion each. Explain *why* the change had the effect you observed, or why it had none.

Answer:

 - (add your short answer here)

**Question 12: Outlook**

If you had one more evening and ten times the compute, which single further change would you invest in for the biggest quality gain, and why?

Answer:

 - (add your short answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)